# Exp 7 — does a causal transformer share machinery across serializations of the same tree?

Successor to Exp 6 (nb6). Same grammar (`build_grammar(seed=0)`), same three orders, same TinyRotaryGPT; multilingual frame dropped.
**This notebook builds E1, E2, E4 only** (handoff run order, step 1). E3 (NoPE), E5 (bidirectional), E6 (sharing push) are *not* built:
the handoff's own stop condition says report E1 before spending training compute, and E5's design needs a decision first.

**Read-before-running: where this notebook departs from the handoff, and why**
- nb6's real models are **12 layers** (not 2), with **two d=96 families**: big-pool `n_head=4` (seeds 0/7/13) and single-point `n_head=12` (seeds 0/7). `n_head` is not stored in the checkpoint, so it is set in `CKPTS`. Default = big-pool h4.
- `linearize()` still discards NT alignment. `linearize_with_paths()` (nb6 cell 84, third redefinition) does carry it, and `sample_tree_indexed` assigns a **language-independent instance id**; that id is the E1 alignment key. nb6 defines these three times with different tuple shapes, so here there is exactly one definition.
- Under L_C's reversal the span **end** of an NT is a *different leaf* than in L_A (span start in A = span end in C). E1 therefore aligns by NT *instance* (span end in each language), not by leaf. `alignment_selftest` asserts this, plus: reverse(A)==C, A'==A, spans contiguous, level-6 order identical in A and B.
- A same-tree A→C patch transplants NT identity **and** order-specific parse context, so "vs floor" alone is confounded. E1 adds within-order same-symbol / different-symbol patches (ceiling and floor that need no new training). See the E1 cell header for the index definitions and the limit that remains.
- E4 (A') must be a **new 4-language model**, not an eval on old checkpoints; with A+A' = 70%, A and A' each see half of A's old exposure.
- Evaluation is fp32 with TF32 off. Training used bf16 autocast; damage differences are small and bf16 noise would swamp them.


## 0. Switches, checkpoints, config

In [9]:
import os, sys, math, json, time, pickle, itertools, subprocess
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

# ============================== SWITCHES ==============================
RUN_E1 = True              # interchange intervention, eval only, existing checkpoints   <- run first
RUN_E2 = True              # attention-offset profiles + head ablation, eval only
RUN_E4_DATAGEN = False     # builds the 4-language training pool (minutes-to-hours of pure python)
RUN_E4_LAUNCH  = False     # launches the A' training run in the background (GPU, ~90K steps). Flip deliberately.

DEVICE = torch.device(os.environ.get("EXP_DEVICE", "cuda:2" if torch.cuda.is_available() else "cpu"))
torch.backends.cuda.matmul.allow_tf32 = False      # eval in fp32/no TF32: damage differences are small, bf16/TF32 noise would swamp them
torch.backends.cudnn.allow_tf32 = False
print("device:", DEVICE)

# ============================== CHECKPOINTS ==============================
# n_layer is read from the checkpoint; n_head CANNOT be inferred, so it is stated here. nb6 has two d=96 families:
#   bp_h4_*  : big-pool (500K trees), n_head=4,  seeds 0/7/13, 90K steps   -> the family the nb6 conclusions rest on
#   h12_*    : single-point check, n_head=12, seeds 0 / 7 (seed-7 dir: exp6_single_point_check_seed2)
CKPTS = {
    "bp_h4_d96_s0":  dict(path="./exp6_bigpool_h4/shared_d96_h4_seed0.pt",  n_head=4),
    "bp_h4_d96_s7":  dict(path="./exp6_bigpool_h4/shared_d96_h4_seed7.pt",  n_head=4),
    "bp_h4_d96_s13": dict(path="./exp6_bigpool_h4/shared_d96_h4_seed13.pt", n_head=4),
    "h12_d96_s0":    dict(path="./exp6_single_point_check/shared_d96.pt",       n_head=12),
    "h12_d96_s7":    dict(path="./exp6_single_point_check_seed2/shared_d96.pt", n_head=12),
    "e4_a2_d96_s0":  dict(path="./exp7_e4/shared4_d96_h4_seed0.pt",         n_head=4),   # produced by section E4 below
}
EVAL_MODELS = ["bp_h4_d96_s0", "bp_h4_d96_s7", "bp_h4_d96_s13"]      # models E1/E2 loop over (missing files are skipped, not fatal)

# ============================== EVAL SET ==============================
N_EVAL_TREES = 600
EVAL_SEED = 999            # nb6 used 999 / 300 trees. Rows are one tree each (BOS + leaves + 0-padding), NOT packed streams.

# ============================== E1 ==============================
LEVELS_E1 = (4, 5, 6)
LAYERS_E1 = (2, 5, 8, 10)  # block-OUTPUT indices (0..11). Patching at 11 only changes the readout at that position.
N_REP     = 4              # random recipient instance per tree per rep (reps reuse trees -> reported SEs are optimistic)
WINDOW    = 8              # score CE at positions p .. p+WINDOW-1 after the patched span end p (inside the tree)
PAIRS_E1  = [("L_A", "L_C"), ("L_C", "L_A"), ("L_A", "L_B"), ("L_B", "L_A"), ("L_B", "L_C"), ("L_C", "L_B")]   # (donor, recipient)
MIN_FLOOR_RANGE = 0.02     # nats. If (floor - within-order same-symbol damage) is below this, the normalised index is reported as nan.
                           # PLACEHOLDER: calibrate from the identity/noise rows, then freeze (prereg cell at the end).
# ============================== E2 ==============================
LEVELS_E2 = (4, 5, 6)
K_MAX = 48                 # offset bins 0..K_MAX-1, bin K_MAX = "far"; the BOS key is accounted separately
N_SPLITS = 50              # random tree half-splits for the within-language floor


device: cuda:2


## 1. Grammar, indexed trees, position alignment (torch-free) + self-test

In [2]:
# ---------------- grammar + indexed trees + alignment (torch-free) ----------------
import random, zlib, numpy as np
from collections import defaultdict

LEVELS = 6; NT_SIZE = 4; TERM_SIZE = 4
MIN_DEGREE, MAX_DEGREE, RULE_LEN_CHOICES = 2, 4, [2, 3]
SEQ_LEN = 256
GRAMMAR_SEED = 0            # nb6's check_rules = build_grammar(seed=0): MUST match what the checkpoints were trained on

LANGS = {                    # order matters: BOS ids are TERM_SIZE + position, so 3-lang models keep ids 4,5,6
    "L_A":  set(),
    "L_B":  {4, 5},
    "L_C":  set(range(1, LEVELS + 1)),
    "L_A2": set(),           # E4 duplicate anchor: same order as L_A, different BOS tag (only exists in 4-lang models)
}
BOS_ALL = {n: TERM_SIZE + i for i, n in enumerate(LANGS)}

def build_grammar(seed):                       # verbatim from nb6
    rng = random.Random(seed)
    symbols_by_level = {1: [(1, 0)]}
    for l in range(2, LEVELS + 1):
        symbols_by_level[l] = [(l, i) for i in range(NT_SIZE)]
    terminals = list(range(TERM_SIZE))
    rules = defaultdict(list)
    for l in range(1, LEVELS + 1):
        child_level = l + 1
        child_symbols = terminals if child_level == LEVELS + 1 else symbols_by_level[child_level]
        for sym in symbols_by_level[l]:
            degree = rng.randint(MIN_DEGREE, MAX_DEGREE)
            seen = set(); tries = 0
            while len(rules[sym]) < degree and tries < 200:
                tries += 1
                rule_len = rng.choice(RULE_LEN_CHOICES)
                rhs = tuple(rng.choice(child_symbols) for _ in range(rule_len))
                if rhs in seen: continue
                seen.add(rhs); rules[sym].append(rhs)
    return rules, symbols_by_level

def sample_tree(rules, rng, symbol=(1, 0), level=1):          # verbatim from nb6 (used for E4 training data)
    rhs = rng.choice(rules[symbol]); children = []
    for child_sym in rhs:
        if level == LEVELS: children.append(child_sym)
        else: children.append(sample_tree(rules, rng, symbol=child_sym, level=level + 1))
    return (symbol, children)

def linearize(node, reverse_levels):                          # verbatim from nb6
    if isinstance(node, int): return [node]
    (level, idx), children = node
    order = list(reversed(children)) if level in reverse_levels else children
    tokens = []
    for c in order: tokens.extend(linearize(c, reverse_levels))
    return tokens

# --- indexed version. ONE definition (nb6 redefines these three times with different tuple shapes). ---
# internal node = (symbol=(level, idx), children, inst);  leaf = ("LEAF", token, leaf_id)
# inst = preorder counter, assigned in the CANONICAL tree, so it is language-independent: the same NT instance
# carries the same inst id in L_A, L_B, L_C. That is the alignment key for E1.
def sample_tree_indexed(rules, rng, symbol=(1, 0), level=1, _leaf=None, _inst=None):
    if _leaf is None: _leaf = [0]
    if _inst is None: _inst = [0]
    inst = _inst[0]; _inst[0] += 1
    rhs = rng.choice(rules[symbol]); children = []
    for child_sym in rhs:
        if level == LEVELS:
            children.append(("LEAF", child_sym, _leaf[0])); _leaf[0] += 1
        else:
            children.append(sample_tree_indexed(rules, rng, child_sym, level + 1, _leaf, _inst))
    return (symbol, children, inst)

def linearize_with_paths(node, reverse_levels, _path=()):
    """-> list of (token, canonical_leaf_id, path) in THIS language's order; path = ((level, idx, inst), ...) root->parent-of-leaf"""
    if node[0] == "LEAF":
        return [(node[1], node[2], _path)]
    (level, idx), children, inst = node
    order = list(reversed(children)) if level in reverse_levels else children
    new_path = _path + ((level, idx, inst),)
    out = []
    for c in order: out.extend(linearize_with_paths(c, reverse_levels, new_path))
    return out

def _seed(*a):                       # stable across processes (builtin string hashing is salted per process)
    return zlib.crc32(repr(a).encode())

def build_eval_core(rules, n_trees, seed, seq_len=SEQ_LEN):
    """Same trees, rendered in every language. Row layout: [BOS, leaf tokens..., 0-padding]. Pure python/numpy.
    SE[lang][level][(t, inst)] = (pos_first, pos_last, nt_idx, n_leaves): the contiguous span of that NT instance
    in this language's row. pos_last = the span END = where the model has just finished reading the constituent."""
    rng = random.Random(seed); cap = seq_len - 1
    trees = []
    while len(trees) < n_trees:
        t = sample_tree_indexed(rules, rng)
        if len(linearize_with_paths(t, set())) <= cap: trees.append(t)
    core = dict(n_trees=n_trees, seq_len=seq_len, tokens={}, nreal={}, SE={}, leafpos={})
    for lang, rev in LANGS.items():
        toks = np.zeros((n_trees, seq_len), np.int64); nreal = np.zeros(n_trees, np.int64)
        se = {l: {} for l in range(1, LEVELS + 1)}
        lp = []                                              # lp[t][leaf_id] = position in this language's row
        for t, tree in enumerate(trees):
            lin = linearize_with_paths(tree, rev)
            toks[t, 0] = BOS_ALL[lang]; nreal[t] = len(lin) + 1; d = {}
            for pos, (tok, lid, path) in enumerate(lin):
                p = pos + 1; toks[t, p] = tok; d[lid] = p
                for (lvl, idx, inst) in path:
                    e = se[lvl].get((t, inst))
                    if e is None: se[lvl][(t, inst)] = [p, p, idx, 1]
                    else: e[1] = p; e[3] += 1
            lp.append(d)
        core["tokens"][lang] = toks; core["nreal"][lang] = nreal
        core["SE"][lang] = {l: {k: tuple(v) for k, v in se[l].items()} for l in se}
        core["leafpos"][lang] = lp
    # language-independent donor pools: POOL[level] = list of (t, inst, nt_idx)
    core["POOL"] = {l: [(t, inst, v[2]) for (t, inst), v in core["SE"]["L_A"][l].items()] for l in range(1, LEVELS + 1)}
    core["trees"] = trees
    return core

def alignment_selftest(core):
    """Hard invariants of the E1 position alignment. Run before trusting any patching number."""
    SE = core["SE"]; n = core["n_trees"]
    # 1. L_C is the exact reverse of L_A on terminals
    for t in range(min(n, 50)):
        a = core["tokens"]["L_A"][t, 1:core["nreal"]["L_A"][t]]; c = core["tokens"]["L_C"][t, 1:core["nreal"]["L_C"][t]]
        assert (a[::-1] == c).all(), "L_C is not reverse(L_A)"
    # 2. L_A2 renders identically to L_A apart from BOS
    assert (core["tokens"]["L_A"][:, 1:] == core["tokens"]["L_A2"][:, 1:]).all()
    assert (core["tokens"]["L_A"][:, 0] != core["tokens"]["L_A2"][:, 0]).all()
    # 3. every NT span is a contiguous interval, same size in every language, same NT symbol, same leaf SET
    for lang in LANGS:
        for l in range(1, LEVELS + 1):
            for key, (f, la, idx, cnt) in SE[lang][l].items():
                assert la - f + 1 == cnt, f"span not contiguous: {lang} level {l} {key}"
                f0, la0, idx0, cnt0 = SE["L_A"][l][key]
                assert cnt == cnt0 and idx == idx0
    # 4. span END is a different leaf under reversal: first leaf (A order) of a level-4 span == last position in C
    t, inst = next(iter(k for k in SE["L_A"][4]))
    inv = {p: lid for lid, p in core["leafpos"]["L_A"][t].items()}; invc = {p: lid for lid, p in core["leafpos"]["L_C"][t].items()}
    fa, la_, _, _ = SE["L_A"][4][(t, inst)]; fc, lc, _, _ = SE["L_C"][4][(t, inst)]
    assert inv[fa] == invc[lc], "span start in A must be span end in C (full reversal)"
    # 5. EXPECTED-SHARED CONTROL: level-6 internal order is identical in A and B (B only reverses levels 4,5)
    for t in range(min(n, 50)):
        for (tt, inst), (f, la, idx, cnt) in SE["L_A"][6].items():
            if tt != t: continue
            fb, lb, _, _ = SE["L_B"][6][(t, inst)]
            seqA = sorted(lid for lid, p in core["leafpos"]["L_A"][t].items() if f <= p <= la)
            ordA = [lid for lid, p in sorted(core["leafpos"]["L_A"][t].items(), key=lambda x: x[1]) if f <= p <= la]
            ordB = [lid for lid, p in sorted(core["leafpos"]["L_B"][t].items(), key=lambda x: x[1]) if fb <= p <= lb]
            assert ordA == ordB, "level-6 span order differs between A and B"
    print("alignment self-test PASSED (reverse(A)==C, A2==A, spans contiguous, span-start(A)==span-end(C), level-6 A==B)")

def select_recipients(core, lang, level, rep, seed=0, min_next=1):
    """One NT instance per tree (level `level`), uniformly among instances whose span END has a next token in `lang`.
    -> list of (t, inst, nt_idx). Deterministic in (seed, level, rep, lang)."""
    rng = random.Random(_seed(seed, level, rep, lang))
    by_tree = defaultdict(list)
    for (t, inst), (f, la, idx, cnt) in core["SE"][lang][level].items():
        if la <= core["nreal"][lang][t] - 1 - min_next: by_tree[t].append((inst, idx))
    out = []
    for t in range(core["n_trees"]):
        if by_tree[t]:
            inst, idx = rng.choice(sorted(by_tree[t])); out.append((t, inst, idx))
    return out

def draw_donors(core, level, recs, mode, rep, seed=0):
    """Language-independent donor instances (t', inst') for each recipient (t, inst, nt):
       same_inst: the SAME instance of the SAME tree          (t'=t, inst'=inst)
       same_nt  : another tree, a level-`level` NT with the SAME symbol   (identity right, context wrong)
       diff_nt  : another tree, a level-`level` NT with a DIFFERENT symbol (identity wrong)"""
    if mode == "same_inst": return [(t, inst) for (t, inst, nt) in recs]
    rng = random.Random(_seed(seed, level, rep, "donor", mode)); pool = core["POOL"][level]; out = []
    for (t, inst, nt) in recs:
        while True:
            t2, i2, nt2 = pool[rng.randrange(len(pool))]
            if t2 != t and ((nt2 == nt) if mode == "same_nt" else (nt2 != nt)): out.append((t2, i2)); break
    return out


## 2. Model, loader, eval helpers

In [3]:
# ---- model: verbatim from nb6 (TinyRotaryGPT, RoPE on q/k). forward returns (logits, loss). ----
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=1024, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)
    def forward(self, seq_len):
        return self.cos[:, :, :seq_len, :], self.sin[:, :, :seq_len, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head; self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd); self.proj = nn.Linear(n_embd, n_embd)
        self.rotary = RotaryEmbedding(self.head_dim, max_seq_len)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rotary(T)
        q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd); self.attn = CausalSelfAttention(n_embd, n_head, max_seq_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))
    def forward(self, x):
        x = x + self.attn(self.ln1(x)); x = x + self.mlp(self.ln2(x)); return x

class TinyRotaryGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_layer=12, n_head=12, max_seq_len=512):
        super().__init__()
        assert n_embd % n_head == 0 and (n_embd // n_head) % 2 == 0, "head_dim must be even for RoPE"
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_seq_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.head = nn.Linear(n_embd, vocab_size, bias=False)
    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for blk in self.blocks: x = blk(x)
        logits = self.head(self.ln_f(x)); loss = None
        if targets is not None: loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

def load_model(path, n_head, seq_len=SEQ_LEN):
    """vocab, d_model, n_layer are read off the checkpoint. 3-lang models (vocab 7) -> A,B,C; 4-lang (vocab 8) -> A,B,C,A2."""
    ck = torch.load(path, map_location="cpu", weights_only=False); sd = ck["model_state"]
    vocab, d = sd["tok_emb.weight"].shape
    n_layer = 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks."))
    m = TinyRotaryGPT(vocab, d, n_layer, n_head, seq_len + 1); m.load_state_dict(sd); m.to(DEVICE).eval()
    langs = list(LANGS)[: vocab - TERM_SIZE]
    tail = ck["losses"][-1000:]
    print(f"  {path}: vocab={vocab} d={d} layers={n_layer} heads={n_head} step={ck['step']} tail-1000 loss={sum(tail)/len(tail):.4f} langs={langs}")
    return m, langs

@torch.no_grad()
def hidden_states(model, tokens, layers, bs=128):
    acts = {l: [] for l in layers}
    hs = [model.blocks[l].register_forward_hook(lambda m, i, o, l=l: acts[l].append(o.detach())) for l in layers]
    try:
        for i in range(0, tokens.shape[0], bs): model(tokens[i:i + bs])
    finally:
        for h in hs: h.remove()
    return {l: torch.cat(acts[l]) for l in layers}                      # {layer: (N, T, d)} on DEVICE

def _ce(logits, x):
    return F.cross_entropy(logits[:, :-1].reshape(-1, logits.shape[-1]).float(), x[:, 1:].reshape(-1),
                           reduction="none").view(x.shape[0], -1)         # (B, T-1); position p predicts token p+1

@torch.no_grad()
def mean_ce_tree(model, tokens, nreal, bs=128, pre_hook_layer=None, pre_hook=None):
    """Mean next-token CE over in-tree targets only (padding excluded). Optional forward-pre-hook on one module."""
    ar = torch.arange(tokens.shape[1] - 1, device=DEVICE); tot = 0.0; cnt = 0.0
    for i in range(0, tokens.shape[0], bs):
        x = tokens[i:i + bs]; nr = nreal[i:i + bs]; h = None
        if pre_hook is not None: h = pre_hook_layer.register_forward_pre_hook(pre_hook)
        try: logits, _ = model(x)
        finally:
            if h is not None: h.remove()
        m = (ar[None, :] <= (nr - 2)[:, None]).float(); tot += (_ce(logits, x) * m).sum().item(); cnt += m.sum().item()
    return tot / cnt

def make_eval_tensors(core):
    tok = {lg: torch.from_numpy(core["tokens"][lg]).to(DEVICE) for lg in LANGS}
    nreal = {lg: torch.from_numpy(core["nreal"][lg]).to(DEVICE) for lg in LANGS}
    return tok, nreal


## 3. E1 — interchange intervention (eval only, existing checkpoints) — RUN FIRST

In [4]:
# =============================== E1: interchange intervention ===============================
# Patch the RESIDUAL STREAM (block-k output) at a span END of a level-l NT instance in the recipient run, with a vector
# taken from a donor run (possibly another language) at that instance's span end, then measure the recipient's CE on the
# next WINDOW tokens relative to its own unpatched run. damage = patched CE - clean CE (nats); bigger = worse.
#
# Why the handoff's "same tree A->C vs floor" is not enough on its own: the span-end residual carries (i) the NT identity
# AND (ii) order-specific parse context (in A the earlier siblings are the LEFT siblings, in C the RIGHT ones). A same-tree
# cross-order patch transplants both. So we ALSO measure, in both orders, patches from OTHER trees with the same / a
# different NT symbol. Within-order same-symbol damage C0 = "identity right, context wrong, format right" is a ceiling that
# needs no extra training; within-order different-symbol damage F = "identity wrong" is the floor.
#   S_floor = 1 - X/F                     X = damage of same-tree cross-order patch      (the handoff's quantity, vs floor only)
#   S_nt    = 1 - (XN - C0)/(F - C0)      XN = damage of cross-order same-symbol patch;  1 = as harmless as within-order, 0 = as harmful as wrong symbol
# Limit that remains: whole-residual patching cannot separate "NT-identity format differs" from "context format differs".

@torch.no_grad()
def window_ce(model, tokens, rec_t, rec_pos, nreal, W, k=None, vecs=None, bs=128):
    n = rec_t.shape[0]; win, nxt = [], []; ar = torch.arange(tokens.shape[1] - 1, device=DEVICE)
    for i in range(0, n, bs):
        t = rec_t[i:i + bs]; p = rec_pos[i:i + bs]; x = tokens[t]; B = x.shape[0]; h = None
        if vecs is not None:
            v = vecs[i:i + bs]; ab = torch.arange(B, device=DEVICE)
            def hook(m, inp, out, v=v, p=p, ab=ab):
                out = out.clone(); out[ab, p] = v.to(out.dtype); return out
            h = model.blocks[k].register_forward_hook(hook)
        try: logits, _ = model(x)
        finally:
            if h is not None: h.remove()
        ce = _ce(logits, x)
        m = ((ar[None, :] >= p[:, None]) & (ar[None, :] < p[:, None] + W) & (ar[None, :] <= (nreal[t] - 2)[:, None])).float()
        win.append((ce * m).sum(1) / m.sum(1).clamp(min=1)); nxt.append(ce[torch.arange(B, device=DEVICE), p])
    return torch.cat(win), torch.cat(nxt)

def run_E1(model, langs, core, tok, nreal, pairs=PAIRS_E1, levels=LEVELS_E1, layers=LAYERS_E1, n_rep=N_REP, W=WINDOW):
    pairs = [(d, r) for d, r in pairs if d in langs and r in langs]
    if "L_A2" in langs: pairs += [("L_A2", "L_A"), ("L_A", "L_A2")]              # E4 ceiling pair, present only in the 4-language model
    recips = sorted({r for _, r in pairs}); donors_of = {r: [d for d, r2 in pairs if r2 == r] for r in recips}
    H = {lg: hidden_states(model, tok[lg], layers) for lg in langs}
    SE = core["SE"]; stats = {}
    for lg in langs:
        for l in levels:
            items = list(SE[lg][l].items())
            ts = torch.tensor([k[0] for k, _ in items], device=DEVICE); ps = torch.tensor([v[1] for _, v in items], device=DEVICE)
            for k in layers:
                X = H[lg][k][ts, ps]; stats[(lg, l, k)] = (X.mean(0), X.std(0))
    acc = {}
    for l in levels:
        for rep in range(n_rep):
            for R in recips:
                recs = select_recipients(core, R, l, rep)
                rt = torch.tensor([r[0] for r in recs], device=DEVICE); rp = torch.tensor([SE[R][l][(r[0], r[1])][1] for r in recs], device=DEVICE)
                cw, cn = window_ce(model, tok[R], rt, rp, nreal[R], W)
                dn = {m: draw_donors(core, l, recs, m, rep) for m in ("same_inst", "same_nt", "diff_nt")}
                def dvec(D, mode, k):
                    dl = dn[mode]; dt = torch.tensor([x[0] for x in dl], device=DEVICE)
                    dp = torch.tensor([SE[D][l][x][1] for x in dl], device=DEVICE); return H[D][k][dt, dp]
                for k in layers:
                    mu, sd = stats[(R, l, k)]; n = rt.shape[0]
                    g = torch.Generator(device=DEVICE); g.manual_seed(_seed(l, rep, R, k))
                    conds = {"ident": H[R][k][rt, rp], "noise": mu + sd * torch.randn(n, mu.shape[0], generator=g, device=DEVICE),
                             "mean": mu.expand(n, -1)}
                    for mode in ("same_nt", "diff_nt"): conds[f"{R}<-{R}:{mode}"] = dvec(R, mode, k)
                    for D in donors_of[R]:
                        for mode in ("same_inst", "same_nt", "diff_nt"): conds[f"{R}<-{D}:{mode}"] = dvec(D, mode, k)
                    for name, vecs in conds.items():
                        w, nx = window_ce(model, tok[R], rt, rp, nreal[R], W, k=k, vecs=vecs)
                        acc.setdefault((R, l, k, name), []).append(((w - cw).cpu(), (nx - cn).cpu()))
    res = {}
    for key, lst in acc.items():
        w = torch.cat([a for a, _ in lst]); x = torch.cat([b for _, b in lst])
        res[key] = dict(win=w.mean().item(), win_se=(w.std() / len(w) ** 0.5).item(), nxt=x.mean().item(), nxt_se=(x.std() / len(x) ** 0.5).item(), n=len(w))
    ident_max = max(abs(v["win"]) for k, v in res.items() if k[3] == "ident")
    print(f"  identity-patch max |damage| = {ident_max:.2e} nats (must be ~0; if > 1e-3 the patching plumbing or numerics are wrong)")
    assert ident_max < 1e-3, "identity patch changed the CE: do not trust any number below"
    return dict(res=res, pairs=pairs, donors_of=donors_of, recips=recips, levels=list(levels), layers=list(layers))

def report_E1(out, tag="", min_range=MIN_FLOOR_RANGE):
    res, nan = out["res"], float("nan")
    for R in out["recips"]:
        for l in out["levels"]:
            print(f"\n[{tag}] recipient {R}, level {l}   damage = patched CE - clean CE over next {WINDOW} tokens (nats)")
            hdr = f"{'layer':>5s} | {'noise':>6s} {'mean':>6s} | {'F=R<-R diff_nt':>14s} {'C0=R<-R same_nt':>15s} |"
            for D in out["donors_of"][R]: hdr += f" {D+':X':>9s} {'XN':>6s} {'S_floor':>7s} {'S_nt':>6s} |"
            print(hdr)
            for k in out["layers"]:
                g = lambda n: res[(R, l, k, n)]["win"]
                F_, C0 = g(f"{R}<-{R}:diff_nt"), g(f"{R}<-{R}:same_nt")
                line = f"{k:>5d} | {g('noise'):6.3f} {g('mean'):6.3f} | {F_:14.3f} {C0:15.3f} |"
                for D in out["donors_of"][R]:
                    X, XN = g(f"{R}<-{D}:same_inst"), g(f"{R}<-{D}:same_nt")
                    s_floor = 1 - X / F_ if F_ > min_range else nan
                    s_nt = 1 - (XN - C0) / (F_ - C0) if (F_ - C0) > min_range else nan
                    line += f" {X:9.3f} {XN:6.3f} {s_floor:7.2f} {s_nt:6.2f} |"
                print(line)
    print("\nS=1: cross-order patch as harmless as the within-order reference;  S=0: as harmful as a wrong-symbol patch;  nan: floor too close to reference (no dynamic range).")
    print("Level 6 is the expected-shared control (A and B read level-6 spans identically). If S is low there too, suspect the instrument, not the model.")


In [5]:
core = build_eval_core(build_grammar(GRAMMAR_SEED)[0], N_EVAL_TREES, EVAL_SEED)
alignment_selftest(core)
tok, nreal = make_eval_tensors(core)
E1_OUT = {}
if RUN_E1:
    for name in EVAL_MODELS:
        if not os.path.exists(CKPTS[name]["path"]): print(f"[skip] {name}: {CKPTS[name]['path']} not found"); continue
        print(f"=== E1 on {name} ==="); model, langs = load_model(CKPTS[name]["path"], CKPTS[name]["n_head"])
        E1_OUT[name] = run_E1(model, langs, core, tok, nreal); report_E1(E1_OUT[name], tag=name)
        del model; torch.cuda.empty_cache() if DEVICE.type == "cuda" else None
    os.makedirs("./exp7_out", exist_ok=True); pickle.dump(E1_OUT, open("./exp7_out/e1_results.pkl", "wb"))
else: print("RUN_E1 is False")


alignment self-test PASSED (reverse(A)==C, A2==A, spans contiguous, span-start(A)==span-end(C), level-6 A==B)
=== E1 on bp_h4_d96_s0 ===
  ./exp6_bigpool_h4/shared_d96_h4_seed0.pt: vocab=7 d=96 layers=12 heads=4 step=90000 tail-1000 loss=0.7973 langs=['L_A', 'L_B', 'L_C']
  identity-patch max |damage| = 0.00e+00 nats (must be ~0; if > 1e-3 the patching plumbing or numerics are wrong)

[bp_h4_d96_s0] recipient L_A, level 4   damage = patched CE - clean CE over next 8 tokens (nats)
layer |  noise   mean | F=R<-R diff_nt C0=R<-R same_nt |     L_C:X     XN S_floor   S_nt |     L_B:X     XN S_floor   S_nt |
    2 |  0.348  0.386 |          0.553           0.511 |     0.645  0.622   -0.17  -1.65 |     0.622  0.643   -0.12  -2.14 |
    5 |  0.138  0.039 |          0.174           0.139 |     0.399  0.398   -1.30  -6.43 |     0.353  0.353   -1.03  -5.14 |
    8 |  0.001 -0.013 |          0.009           0.002 |     0.022  0.024     nan    nan |     0.047  0.048     nan    nan |
   10 | -0.012 

## 4. E2 — attention-offset profiles + head ablation (eval only)

In [6]:
# =============================== E2: attention-offset profiles + head ablation (M1) ===============================
# For every head: how much attention mass sits at each relative offset, measured at the span ENDS of level-l NT instances,
# separately per language on the SAME trees. Then: do profiles differ across languages (vs a within-language split floor),
# and are the heads that differ the ones whose ablation matters? Single model only; never compare head indices across seeds.

def attn_probs(attn, x):
    B, T, C = x.shape; H, hd = attn.n_head, attn.head_dim
    qkv = attn.qkv(x).view(B, T, 3, H, hd).permute(2, 0, 3, 1, 4); q, k = qkv[0], qkv[1]
    cos, sin = attn.rotary(T); q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
    s = (q @ k.transpose(-1, -2)) / math.sqrt(hd)
    s = s.masked_fill(torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), 1), float("-inf"))
    return s.float().softmax(-1)                                                   # (B, H, T, T)

@torch.no_grad()
def offset_profiles(model, core, tok, lang, level, bs=64):
    """mass[t, layer, head, bin]: attention mass from level-`level` span-end queries of tree t, binned by offset (query - key);
    BOS key excluded from the bins and returned separately; bin K_MAX = offsets >= K_MAX."""
    L = len(model.blocks); H = model.blocks[0].attn.n_head; N = tok.shape[0]
    mass = torch.zeros(L, N, H, K_MAX + 1, device=DEVICE); bosm = torch.zeros(L, N, H, device=DEVICE)
    items = list(core["SE"][lang][level].items())
    qt_all = torch.tensor([k[0] for k, _ in items]); qp_all = torch.tensor([v[1] for _, v in items]); ctx = {}
    def make_pre(l):
        def pre(mod, args):
            P = attn_probs(mod, args[0]); qb, qp, qt = ctx["qb"], ctx["qp"], ctx["qt"]
            rows = P[qb, :, qp, :]                                                  # (Q, H, T)
            ar = torch.arange(rows.shape[-1], device=DEVICE); off = qp[:, None] - ar[None, :]
            valid = ((off >= 0) & (ar[None, :] >= 1)).float(); bins = off.clamp(0, K_MAX)
            m = torch.zeros(rows.shape[0], H, K_MAX + 1, device=DEVICE).scatter_add_(2, bins[:, None, :].expand(-1, H, -1), rows * valid[:, None, :])
            mass[l].index_add_(0, qt, m); bosm[l].index_add_(0, qt, rows[:, :, 0])
        return pre
    hs = [model.blocks[l].attn.register_forward_pre_hook(make_pre(l)) for l in range(L)]
    try:
        for s in range(0, N, bs):
            sel = (qt_all >= s) & (qt_all < s + bs)
            if not sel.any(): continue
            ctx["qt"] = qt_all[sel].to(DEVICE); ctx["qp"] = qp_all[sel].to(DEVICE); ctx["qb"] = ctx["qt"] - s
            model(tok[s:s + bs])
    finally:
        for h in hs: h.remove()
    return dict(mass=mass.permute(1, 0, 2, 3).cpu().numpy(), bos=bosm.permute(1, 0, 2).cpu().numpy())

def _emd(p, q): return np.abs(np.cumsum(p - q, -1)).sum(-1)                         # 1-D earth mover's distance over offset bins
def _norm(mass, trees):
    m = mass[trees].sum(0); return m / np.clip(m.sum(-1, keepdims=True), 1e-12, None)   # (L, H, K+1)

def profile_distances(profA, profB, n_split=N_SPLITS, seed=0):
    """cross = mean EMD(A on half 1, B on half 2); floor = EMD(A on half 1, A on half 2). Same half sizes, so noise is matched.
    A head 'differs' if cross > the 95th percentile of its own within-language split floor."""
    rng = np.random.RandomState(seed); N = profA["mass"].shape[0]; cross, floor = [], []
    for _ in range(n_split):
        perm = rng.permutation(N); s1, s2 = perm[: N // 2], perm[N // 2:]
        a1, a2, b2 = _norm(profA["mass"], s1), _norm(profA["mass"], s2), _norm(profB["mass"], s2)
        cross.append(_emd(a1, b2)); floor.append(_emd(a1, a2))
    cross, floor = np.stack(cross), np.stack(floor)
    return dict(cross=cross.mean(0), floor_mean=floor.mean(0), floor95=np.percentile(floor, 95, axis=0))   # each (L, H)

@torch.no_grad()
def head_mean_outputs(model, toks, nreals, bs=128):
    """Language-pooled mean of each head's pre-projection output over in-tree positions (the mean-ablation target).
    Pooled across languages on purpose: a per-language mean would partly bake the language into the ablation."""
    L = len(model.blocks); H = model.blocks[0].attn.n_head; hd = model.blocks[0].attn.head_dim
    sums = torch.zeros(L, H, hd, device=DEVICE); total = 0.0; ctx = {}
    def make(l):
        def pre(mod, args):
            y = args[0]; yy = y.view(y.shape[0], y.shape[1], H, hd)
            sums[l] += (yy * ctx["m"][:, :, None, None]).sum((0, 1))
        return pre
    hs = [model.blocks[l].attn.proj.register_forward_pre_hook(make(l)) for l in range(L)]
    try:
        for tk, nr in zip(toks, nreals):
            ar = torch.arange(tk.shape[1], device=DEVICE)
            for i in range(0, tk.shape[0], bs):
                ctx["m"] = (ar[None, :] < nr[i:i + bs][:, None]).float(); total += ctx["m"].sum().item()
                model(tk[i:i + bs])
    finally:
        for h in hs: h.remove()
    return sums / total

def head_ablation(model, core, tok, nreal, langs):
    L = len(model.blocks); H = model.blocks[0].attn.n_head; hd = model.blocks[0].attn.head_dim
    mu = head_mean_outputs(model, [tok[lg] for lg in langs], [nreal[lg] for lg in langs])
    base = {lg: mean_ce_tree(model, tok[lg], nreal[lg]) for lg in langs}; d = {lg: np.zeros((L, H)) for lg in langs}
    for l in range(L):
        for h in range(H):
            def pre(mod, args, l=l, h=h):
                y = args[0].clone(); y[..., h * hd:(h + 1) * hd] = mu[l, h].to(y.dtype); return (y,)
            for lg in langs:
                d[lg][l, h] = mean_ce_tree(model, tok[lg], nreal[lg], pre_hook_layer=model.blocks[l].attn.proj, pre_hook=pre) - base[lg]
    return dict(base=base, dce=d)

def _rank(a): return np.argsort(np.argsort(a)).astype(float)

def run_E2(model, langs, core, tok, nreal, levels=LEVELS_E2, comparisons=(("L_A", "L_C"), ("L_A", "L_B"))):
    prof = {(lg, l): offset_profiles(model, core, tok[lg], lg, l) for lg in langs for l in levels}
    dist = {(a, b, l): profile_distances(prof[(a, l)], prof[(b, l)]) for a, b in comparisons if a in langs and b in langs for l in levels}
    abl = head_ablation(model, core, tok, nreal, langs)
    return dict(prof_bos={k: v["bos"].mean(0) for k, v in prof.items()}, dist=dist, abl=abl, levels=list(levels))

def report_E2(out, tag="", top_frac=0.25, n_show=12):
    abl = out["abl"]
    langs = list(abl["dce"]); mx = np.max([abl["dce"][lg] for lg in langs], axis=0)       # (L, H): importance = worst-case language
    matter = mx >= np.quantile(mx, 1 - top_frac)
    print(f"\n[{tag}] head ablation (language-pooled mean ablation): CE change per language; baseline CE: " + ", ".join(f"{lg} {abl['base'][lg]:.3f}" for lg in langs))
    order = np.dstack(np.unravel_index(np.argsort(-mx, axis=None), mx.shape))[0][:n_show]
    print(f"  top {n_show} heads by max dCE:   (layer,head)  " + "  ".join(f"d{lg[-1]}" for lg in langs))
    for l, h in order: print(f"    L{l:<2d}H{h}   " + "  ".join(f"{abl['dce'][lg][l, h]:+.3f}" for lg in langs))
    print(f"\n  profile distance A vs other, 'differs' = cross EMD > 95th pct of the within-language split floor.  'matter' = top {int(top_frac*100)}% of heads by max dCE")
    print(f"  {'cmp':>10s} {'lvl':>3s} | {'#differ':>7s} {'differ|matter':>13s} {'differ|other':>12s} | {'spearman(excess EMD, |dCE_b-dCE_a|)':>36s}")
    for (a, b, l), d in sorted(out["dist"].items(), key=lambda kv: (kv[0][1], kv[0][2])):
        differs = d["cross"] > d["floor95"]; excess = (d["cross"] - d["floor_mean"]).ravel()
        spec = np.abs(abl["dce"][b] - abl["dce"][a]).ravel(); r = np.corrcoef(_rank(excess), _rank(spec))[0, 1]
        fm = differs[matter].mean() if matter.any() else float("nan"); fo = differs[~matter].mean() if (~matter).any() else float("nan")
        print(f"  {a[-1]}-vs-{b[-1]:>6s} {l:>3d} | {int(differs.sum()):>3d}/{differs.size:<3d} {fm:13.2f} {fo:12.2f} | {r:36.2f}")
    print("\n  Surprising reading needs BOTH: many heads differ at levels 4-5 (not at 6) AND differ|matter >> differ|other.")
    print("  Caveat: attention at span ends is a tiny slice of what a head does; a head with a language-independent offset profile can still use different content.")

E2_OUT = {}
if RUN_E2:
    for name in EVAL_MODELS:
        if not os.path.exists(CKPTS[name]["path"]): print(f"[skip] {name}"); continue
        print(f"=== E2 on {name} ==="); model, langs = load_model(CKPTS[name]["path"], CKPTS[name]["n_head"])
        E2_OUT[name] = run_E2(model, langs, core, tok, nreal); report_E2(E2_OUT[name], tag=name)
        del model; torch.cuda.empty_cache() if DEVICE.type == "cuda" else None
    os.makedirs("./exp7_out", exist_ok=True); pickle.dump(E2_OUT, open("./exp7_out/e2_results.pkl", "wb"))
else: print("RUN_E2 is False")


=== E2 on bp_h4_d96_s0 ===


  ./exp6_bigpool_h4/shared_d96_h4_seed0.pt: vocab=7 d=96 layers=12 heads=4 step=90000 tail-1000 loss=0.7973 langs=['L_A', 'L_B', 'L_C']

[bp_h4_d96_s0] head ablation (language-pooled mean ablation): CE change per language; baseline CE: L_A 0.772, L_B 0.778, L_C 0.782
  top 12 heads by max dCE:   (layer,head)  dA  dB  dC
    L3 H1   +0.144  +0.158  +0.145
    L7 H1   +0.004  +0.017  +0.133
    L2 H3   +0.132  +0.103  +0.112
    L0 H2   +0.117  +0.109  +0.035
    L11H1   +0.011  +0.020  +0.008
    L1 H2   +0.017  +0.011  +0.003
    L4 H2   +0.016  +0.004  +0.001
    L9 H0   +0.001  +0.003  +0.015
    L9 H2   +0.004  +0.004  +0.013
    L5 H2   +0.012  +0.012  +0.000
    L8 H3   +0.012  +0.002  +0.001
    L11H2   +0.008  +0.008  +0.011

  profile distance A vs other, 'differs' = cross EMD > 95th pct of the within-language split floor.  'matter' = top 25% of heads by max dCE
         cmp lvl | #differ differ|matter differ|other |  spearman(excess EMD, |dCE_b-dCE_a|)
  A-vs-     B   4 |  44/

## 5. E4 — A' calibration run (training, background)

In [7]:
# =============================== E4: A' calibration run (duplicate anchor) ===============================
# A' = same linearization as A, different BOS tag. It only means something if the model is TRAINED with it, so this is a new
# 4-language model (vocab 8), not an extra eval on the old checkpoints. Mix keeps A+A' = 70% so B and C keep exactly 15% each;
# consequence: A and A' each see HALF the sequences that A saw in the 3-language runs. Compare A vs A' inside this one model;
# comparing this model's A to the old models' A carries that mild confound. Everything else = nb6 big-pool recipe:
# d=96, n_head=4, 12 layers, 500K-tree pool, batch 32, lr 3e-3 AdamW, 90K steps fixed (no convergence flag), bf16 autocast.
E4 = dict(root="./exp7_e4", big_n=500_000, chunk=50_000, d=96, n_head=4, seed=0, data_seed=1, steps=90_000,
          mix={"L_A": 0.35, "L_A2": 0.35, "L_B": 0.15, "L_C": 0.15})
assert abs(E4["mix"]["L_A"] + E4["mix"]["L_A2"] - 0.70) < 1e-9 and E4["mix"]["L_B"] == 0.15 and E4["mix"]["L_C"] == 0.15

def make_mix_stream(rules, n_trees, seed, seq_len, mix):
    rng = random.Random(seed); names = list(mix); w = [mix[n] for n in names]; stream = []
    for _ in range(n_trees):
        lang = rng.choices(names, weights=w)[0]; tree = sample_tree(rules, rng)
        stream.append(BOS_ALL[lang]); stream.extend(linearize(tree, LANGS[lang]))
    s = torch.tensor(stream, dtype=torch.long); n = len(s) // (seq_len + 1)
    return s[: n * (seq_len + 1)].view(n, seq_len + 1)

def gen_pool(make_fn, n_total, chunk, seed0):
    parts, t0 = [], time.time()
    for i, start in enumerate(range(0, n_total, chunk)):
        n = min(chunk, n_total - start); parts.append(make_fn(n, seed0 * 1000 + i).cpu())
        if i == 0: el = time.time() - t0; print(f"    first chunk ({n} trees): {el:.0f}s -> est. total {el * n_total / n / 60:.1f} min", flush=True)
    return torch.cat(parts)

E4_DATA = os.path.join(E4["root"], f"data_shared4_seed{E4['data_seed']}.pt")
E4_CKPT = CKPTS["e4_a2_d96_s0"]["path"]
if RUN_E4_DATAGEN and not os.path.exists(E4_DATA):
    os.makedirs(E4["root"], exist_ok=True); rules_e4, _ = build_grammar(GRAMMAR_SEED)
    pool = gen_pool(lambda n, sd: make_mix_stream(rules_e4, n, sd, SEQ_LEN, E4["mix"]), E4["big_n"], E4["chunk"], E4["data_seed"])
    torch.save(pool, E4_DATA); print(f"saved {tuple(pool.shape)} -> {E4_DATA}")
    # the pool must contain all four tags in the intended proportions
    first = pool[:, 0]; print({n: round(float((first == b).float().mean()), 3) for n, b in BOS_ALL.items()}, "(row-start tags only; rows are packed streams)")

# worker: verbatim copy of nb6's train_worker.py (written only if absent, so an existing worker is reused untouched)
WORKER_SRC = r'''import argparse, os, time
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- model: verbatim copy of the notebook's model cell ----
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=1024, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)

    def forward(self, seq_len):
        return self.cos[:, :, :seq_len, :], self.sin[:, :, :seq_len, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.rotary = RotaryEmbedding(self.head_dim, max_seq_len)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rotary(T)
        q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_seq_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyRotaryGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_layer=12, n_head=12, max_seq_len=512):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        assert (n_embd // n_head) % 2 == 0, "head_dim must be EVEN for rotary embeddings"
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_seq_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

# ---- training: same recipe as train_fixed_budget ----
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--d", type=int, required=True)
    ap.add_argument("--seed", type=int, required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--vocab", type=int, required=True)
    ap.add_argument("--seq_len", type=int, required=True)
    ap.add_argument("--steps", type=int, default=90000)
    ap.add_argument("--n_head", type=int, default=4)
    ap.add_argument("--n_layer", type=int, default=12)
    ap.add_argument("--device", default="cuda:0")
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--lr", type=float, default=3e-3)
    ap.add_argument("--compile", type=int, default=0)
    a = ap.parse_args()

    torch.set_num_threads(1)                     # avoid CPU oversubscription with several workers
    device = torch.device(a.device)
    torch.manual_seed(a.seed)
    data = torch.load(a.data).to(device)
    model = TinyRotaryGPT(a.vocab, n_embd=a.d, n_layer=a.n_layer, n_head=a.n_head,
                          max_seq_len=a.seq_len + 1).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, fused=True)
    fwd = torch.compile(model) if a.compile else model
    n = data.shape[0]
    buf = torch.zeros(a.steps, device=device)
    t0 = time.time()
    for step in range(a.steps):
        idx = torch.randint(0, n, (a.batch,), device=device)
        batch = data[idx]
        x, y = batch[:, :-1], batch[:, 1:]
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            _, loss = fwd(x, y)
        opt.zero_grad(); loss.backward(); opt.step()
        buf[step] = loss.detach().float()
        if (step + 1) % 5000 == 0:
            print(f"step {step+1:>7d}: mean loss over last 5000 = {buf[step-4999:step+1].mean().item():.4f}"
                  f"  ({(step + 1) / (time.time() - t0):.0f} it/s)", flush=True)
        if (step + 1) % 10000 == 0 or step == a.steps - 1:
            tmp = a.ckpt + ".tmp"
            torch.save({"model_state": model.state_dict(), "opt_state": opt.state_dict(),
                        "step": step + 1, "losses": buf[:step + 1].tolist()}, tmp)
            os.replace(tmp, a.ckpt)

if __name__ == "__main__":
    main()'''
if not os.path.exists("train_worker.py"):
    open("train_worker.py", "w").write(WORKER_SRC); print("wrote train_worker.py (copy of nb6's worker)")

def ckpt_step(path):
    if not os.path.exists(path): return -1
    return torch.load(path, map_location="cpu", weights_only=False)["step"]

E4_PROC = None
if RUN_E4_LAUNCH:
    assert os.path.exists(E4_DATA), "run with RUN_E4_DATAGEN=True first"
    if ckpt_step(E4_CKPT) == E4["steps"]: print("E4 checkpoint already complete:", E4_CKPT)
    else:
        os.makedirs(E4["root"], exist_ok=True); log = open(os.path.join(E4["root"], "log_e4_a2_d96_s0.txt"), "w")
        cmd = [sys.executable, "train_worker.py", "--d", str(E4["d"]), "--seed", str(E4["seed"]), "--data", E4_DATA, "--ckpt", E4_CKPT,
               "--vocab", str(len(BOS_ALL) + TERM_SIZE), "--seq_len", str(SEQ_LEN), "--steps", str(E4["steps"]),
               "--n_head", str(E4["n_head"]), "--device", str(DEVICE)]
        E4_PROC = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
        print("launched E4 in the background, pid", E4_PROC.pid, "| log:", log.name, "| ckpt:", E4_CKPT)
        print("When it finishes: add 'e4_a2_d96_s0' to EVAL_MODELS and re-run the E1 cell. E1 then also reports L_A2->L_A (the A' ceiling).")
else: print("RUN_E4_LAUNCH is False -- nothing launched.")


RUN_E4_LAUNCH is False -- nothing launched.


## 6. Pre-registration and verdict (refuses to give a verdict until thresholds are frozen)

In [8]:
# =============================== PRE-REGISTRATION (E1, E2) ===============================
# Thresholds are deliberately None. Fill them from the calibration rows (identity / noise / mean, within-order C0 vs F, and
# the L_A2->L_A ceiling once E4 exists), then call freeze_thresholds(). verdict_E1 refuses to speak on unfrozen thresholds,
# so an ambiguous number cannot be read generously after the fact. The layer used for a verdict is fixed by RULE (the layer
# with the largest F - C0 dynamic range for that recipient, averaged over levels 4-5), not by looking at S.
PREREG_TABLE = """
| Exp | Surprising reading                                                                              | Boring reading                                                       |
| E1  | S_nt(A<->C) near floor at levels 4-5, S_nt(A2->A) near ceiling, level 6 (A/B) stays high       | S_nt(A<->C) not near floor, or no dynamic range (F - C0 tiny)         |
| E2  | many heads' offset profiles differ A-vs-C at levels 4-5, AND differ|matter >> differ|other      | profiles within the split floor, or differ only in heads that don't matter |
"""
THRESH = dict(e1_near_floor_S_nt=None, e1_near_ceiling_S_nt=None, e1_min_floor_range=None,
              e2_floor_pct=95, e2_matter_top_frac=0.25, e2_differ_matter_ratio=None)
THRESH_FILE = "./exp7_out/prereg_frozen.json"

def freeze_thresholds():
    missing = [k for k, v in THRESH.items() if v is None]
    assert not missing, f"fill these first: {missing}"
    os.makedirs("./exp7_out", exist_ok=True); json.dump(dict(THRESH, frozen_at=time.strftime("%Y-%m-%d %H:%M:%S")), open(THRESH_FILE, "w"), indent=2)
    print("frozen ->", THRESH_FILE)

def _frozen():
    if not os.path.exists(THRESH_FILE): return None
    j = json.load(open(THRESH_FILE)); return j if all(j.get(k) == v for k, v in THRESH.items()) else None

def verdict_E1(out_by_model):
    print(PREREG_TABLE)
    fz = _frozen()
    if fz is None:
        print("UNFROZEN thresholds -> no verdict. Read the raw tables above; fill THRESH from the calibration rows; freeze_thresholds(); re-run this cell."); return
    nan = float("nan")
    for name, out in out_by_model.items():
        res = out["res"]; g = lambda R, l, k, n: res[(R, l, k, n)]["win"]
        def S_nt(R, D, l, k):
            F_, C0, XN = g(R, l, k, f"{R}<-{R}:diff_nt"), g(R, l, k, f"{R}<-{R}:same_nt"), g(R, l, k, f"{R}<-{D}:same_nt")
            return 1 - (XN - C0) / (F_ - C0) if (F_ - C0) > fz["e1_min_floor_range"] else nan
        def layer_for(R):
            rng = {k: np.mean([g(R, l, k, f"{R}<-{R}:diff_nt") - g(R, l, k, f"{R}<-{R}:same_nt") for l in (4, 5)]) for k in out["layers"]}
            return max(rng, key=rng.get)
        print(f"--- {name} ---")
        kC, kA = layer_for("L_C"), layer_for("L_A")
        ac = [S_nt("L_C", "L_A", l, kC) for l in (4, 5)]; ca = [S_nt("L_A", "L_C", l, kA) for l in (4, 5)]
        lvl6 = S_nt("L_B", "L_A", 6, layer_for("L_B")) if ("L_A", "L_B") in out["pairs"] else nan
        ceil = S_nt("L_A", "L_A2", 4, kA) if ("L_A2", "L_A") in out["pairs"] else nan
        print(f"  S_nt A->C (recipient C, layer {kC}), levels 4,5: {ac}   C->A (recipient A, layer {kA}): {ca}")
        print(f"  level-6 A->B expected-shared control: {lvl6}     A'->A ceiling: {ceil}")
        near_floor = all(x <= fz["e1_near_floor_S_nt"] for x in ac + ca if x == x)
        ceil_ok = (ceil != ceil) or ceil >= fz["e1_near_ceiling_S_nt"]; l6_ok = (lvl6 != lvl6) or lvl6 >= fz["e1_near_ceiling_S_nt"]
        print("  READING:", "SURPRISING-side (non-sharing at 4-5 with working controls)" if near_floor and ceil_ok and l6_ok
              else ("INSTRUMENT SUSPECT (controls fail)" if near_floor else "BORING-side (A<->C not near floor)"))
        if ceil != ceil: print("  note: no A' ceiling yet (E4 not trained) -> 'SURPRISING-side' is provisional")

if E1_OUT: verdict_E1(E1_OUT)



| Exp | Surprising reading                                                                              | Boring reading                                                       |
| E1  | S_nt(A<->C) near floor at levels 4-5, S_nt(A2->A) near ceiling, level 6 (A/B) stays high       | S_nt(A<->C) not near floor, or no dynamic range (F - C0 tiny)         |
| E2  | many heads' offset profiles differ A-vs-C at levels 4-5, AND differ|matter >> differ|other      | profiles within the split floor, or differ only in heads that don't matter |

UNFROZEN thresholds -> no verdict. Read the raw tables above; fill THRESH from the calibration rows; freeze_thresholds(); re-run this cell.


## Not built here
- **E3 NoPE swap** — needs retraining at d=96, same budget/seeds; do after E1 is clean.
- **E5 bidirectional control** — keep the objective fixed and change only the mask (same cloze/masked-span objective, causal vs full mask). Remaining confounds to settle before building: (i) under a full mask, A and C are exact mirror images, so sharing there can be achieved by sign-flip of relative offsets and says nothing about B; (ii) a full-mask cloze model can solve the task with local neighbours instead of parse-level computation, so a short mask span rate matters.
- **E6 sharing push** — needs aligned NT labels at span ends (available via `SE`) and the per-order-head control; start only once E1 and E4 give a usable ceiling.
- Rho from Pedra Bento (NeurIPS 2026): not implemented; I have not read the paper and will not reconstruct the invariant from memory.
